# Packet P-042 — Panel Validation Within Family

**Decision question:** Are the 3 panel devices still top-ranked when evaluated only against devices in their own composition family? P-037 showed the model is family-dependent — so the relevant question is within-family ranking, not global.

**Fastest falsifier:** Rank panel device 850 (Pure MA) against all Pure MA devices. If not in top-20 of Pure MA, the within-family ranking fails.

**Success:** All 3 panel devices rank in top-20 of their own family across 20 splits.
**Kill:** Any panel device falls below top-50% within its family.

In [1]:
"""Cell 1 — Panel within-family ranking across 20 splits."""
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split
from scipy.stats import kendalltau
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('perovskite_stability_clean.csv')
FEATURES = [
    'Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
    'MA', 'FA', 'Cs',
    'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
    'Perovskite_thickness', 'Cell_area_measured',
    'JV_default_Voc', 'JV_default_Jsc', 'JV_default_FF'
]
TARGET = 'Stability_PCE_T80'
ET_PARAMS = dict(n_estimators=700, max_features='sqrt', min_samples_split=3,
                 min_samples_leaf=1, bootstrap=False, random_state=42)

X = df[FEATURES].fillna(0)
y = np.log1p(df[TARGET])

def classify_family(row):
    ma, fa, cs = row["MA"] > 0, row["FA"] > 0, row["Cs"] > 0
    if ma and not fa and not cs: return "Pure MA"
    elif fa and not ma and not cs: return "Pure FA"
    elif ma and fa and not cs: return "MA-FA mixed"
    elif fa and cs and not ma: return "FA-Cs"
    elif ma and fa and cs: return "Triple cation"
    else: return "Other"

df["family"] = df.apply(classify_family, axis=1)

# Panel devices and their families
PANEL = {
    850: 'Pure MA',      # MA₃Pb₄I₁₃
    1320: 'MA-FA mixed', # MA₀.₂₅FA₀.₇₅PbI₂.₇₇Br₀.₂₅
    119: 'FA-Cs',        # FA₀.₈₃Cs₀.₁₇PbI₂Br
}

# Verify family assignments
for d, expected_fam in PANEL.items():
    actual_fam = df.loc[d, "family"]
    print(f"Device {d}: expected={expected_fam}, actual={actual_fam}")

N_SPLITS = 20
panel_results = {d: {'appearances': 0, 'top20_family': 0, 'top50pct_family': 0, 
                       'percentile': []} for d in PANEL}

for seed in range(N_SPLITS):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed)
    
    model = ExtraTreesRegressor(**ET_PARAMS)
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    
    test_idx = X_te.index.tolist()
    test_families = df.loc[X_te.index, "family"]
    
    for d, fam in PANEL.items():
        if d not in test_idx:
            continue
        
        panel_results[d]['appearances'] += 1
        
        # Get all test devices in same family
        fam_mask = test_families == fam
        fam_idx = [i for i in test_idx if df.loc[i, "family"] == fam]
        fam_preds = {i: preds[test_idx.index(i)] for i in fam_idx}
        
        # Rank within family (descending by prediction)
        ranked = sorted(fam_preds.items(), key=lambda x: -x[1])
        rank_pos = [i for i, (idx, _) in enumerate(ranked) if idx == d][0]
        n_fam = len(ranked)
        percentile = 1 - rank_pos / n_fam  # 1.0 = top, 0.0 = bottom
        
        panel_results[d]['percentile'].append(percentile)
        
        if rank_pos < 20:
            panel_results[d]['top20_family'] += 1
        if percentile >= 0.5:
            panel_results[d]['top50pct_family'] += 1

    if (seed + 1) % 10 == 0:
        print(f"  Completed {seed+1}/{N_SPLITS} splits")

# Report
print(f"\n{'=' * 70}")
print("PANEL WITHIN-FAMILY RANKING")
print(f"{'=' * 70}")
print(f"{'Device':<10} {'Family':<15} {'Apps':>5} {'Top-20 fam':>12} {'Top-50%':>10} {'Mean pctile':>12}")
print("-" * 65)

all_pass = True
for d, fam in PANEL.items():
    r = panel_results[d]
    apps = r['appearances']
    t20 = r['top20_family']
    t50 = r['top50pct_family']
    mean_p = np.mean(r['percentile']) if r['percentile'] else float('nan')
    rate_20 = f"{t20}/{apps}" if apps > 0 else "N/A"
    rate_50 = f"{t50}/{apps}" if apps > 0 else "N/A"
    print(f"{d:<10} {fam:<15} {apps:>5} {rate_20:>12} {rate_50:>10} {mean_p:>12.1%}")
    if apps > 0 and t50 / apps < 0.80:
        all_pass = False

Device 850: expected=Pure MA, actual=Pure MA
Device 1320: expected=MA-FA mixed, actual=MA-FA mixed
Device 119: expected=FA-Cs, actual=FA-Cs


  Completed 10/20 splits


  Completed 20/20 splits

PANEL WITHIN-FAMILY RANKING
Device     Family           Apps   Top-20 fam    Top-50%  Mean pctile
-----------------------------------------------------------------
850        Pure MA             4          4/4        4/4       100.0%
1320       MA-FA mixed         3          3/3        3/3        92.2%
119        FA-Cs               4          4/4        4/4        92.0%


In [2]:
"""Cell 2 — Evaluate and save."""

# Status: pass if all panel devices are top-50% within family >=80% of time
if all_pass:
    status = "Confirmed"
else:
    # Check if any panel device is consistently bottom-half
    any_bottom = False
    for d in PANEL:
        r = panel_results[d]
        if r['appearances'] > 0 and r['top50pct_family'] / r['appearances'] < 0.50:
            any_bottom = True
    status = "Negative" if any_bottom else "Promising"

save_rows = []
for d, fam in PANEL.items():
    r = panel_results[d]
    save_rows.append({
        'device': d, 'family': fam,
        'appearances': r['appearances'],
        'top20_family_rate': r['top20_family'] / r['appearances'] if r['appearances'] > 0 else float('nan'),
        'top50pct_rate': r['top50pct_family'] / r['appearances'] if r['appearances'] > 0 else float('nan'),
        'mean_percentile': np.mean(r['percentile']) if r['percentile'] else float('nan'),
    })
pd.DataFrame(save_rows).to_csv('Packet_P042_Panel_Within_Family.csv', index=False)
print(f"Saved: Packet_P042_Panel_Within_Family.csv")

print(f"\n{'=' * 60}")
print(f"P-042 STATUS: {status}")
print(f"{'=' * 60}")
if status == "Confirmed":
    print("All panel devices rank well within their own family.")
    print("Even under P-037's family-dependent model, panel selection is valid.")
elif status == "Negative":
    print("At least one panel device is not well-ranked within its family.")
else:
    print("Mixed within-family ranking — some panel devices are borderline.")

Saved: Packet_P042_Panel_Within_Family.csv

P-042 STATUS: Confirmed
All panel devices rank well within their own family.
Even under P-037's family-dependent model, panel selection is valid.
